In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
from texas_gerrymandering_hb4.config import GEO_VTD

TITLE = "Texas demographics"


In [6]:
geo = pd.read_parquet(GEO_VTD)
geo.columns = [c.strip() for c in geo.columns]
geo.head()

,vtd_geoid,total_pop,total_nh_white,total_nh_black,total_hisp,total_nh_asian,vap_total,vap_nh_white,vap_nh_black,vap_hisp,vap_nh_asian,cvap_total,cvap_hisp,cvap_nh_white,cvap_nh_black,total_other,vap_other,cvap_other,state_fips
0,1.0,3162,1663,914,491,53,2348,1350,632,287,40,2544,81,1598,760,94,79,105,48
1,2.0,3811,3159,256,294,16,2880,2453,172,173,11,3695,279,2662,353,102,82,401,48
2,3.0,1916,1405,150,290,25,1465,1148,97,156,15,1652,40,1293,219,71,64,100,48
3,4.0,2306,2004,91,131,6,1821,1605,63,82,4,2145,58,1716,248,80,71,123,48
4,5.0,405,335,14,37,1,325,274,11,24,1,300,9,231,25,19,16,35,48


In [7]:
def pick_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def sum_col(df, col):
    return int(pd.to_numeric(df[col], errors="coerce").fillna(0).sum())

def fmt_pct(x):
    return f"{x*100:.2f}%"

GROUPS = [
    ("Latino", "hisp"),
    ("Black", "black"),
    ("White", "white"),
    ("Other", "other"),
]

# ---- column detection (edit here if your naming differs) ----

# total population
total_total_col = pick_existing(geo, ["total_pop", "tot_pop", "pop_total", "total"])

total_by_group = {
    "hisp": pick_existing(geo, ["total_hisp", "pop_hisp", "hisp_total", "tot_hisp"]),
    "black": pick_existing(geo, ["total_black", "pop_black", "black_total", "tot_black"]),
    "white": pick_existing(geo, ["total_white", "pop_white", "white_total", "tot_white"]),
    "other": pick_existing(geo, ["total_other", "pop_other", "other_total", "tot_other"]),
}

# VAP
vap_total_col = pick_existing(geo, ["vap_total", "vap"])
vap_by_group = {
    "hisp": pick_existing(geo, ["vap_hisp", "hispvap", "vap_latino"]),
    "black": pick_existing(geo, ["vap_black", "blackvap", "vap_nh_black", "vap_black_nh"]),
    "white": pick_existing(geo, ["vap_white", "whitevap", "vap_nh_white", "vap_white_nh"]),
    "other": pick_existing(geo, ["vap_other", "othervap"]),
}

# CVAP
cvap_total_col = pick_existing(geo, ["cvap_total", "cvap"])
cvap_by_group = {
    "hisp": pick_existing(geo, ["cvap_hisp", "cvap_latino"]),
    "black": pick_existing(geo, ["cvap_nh_black", "cvap_black", "cvap_black_nh"]),
    "white": pick_existing(geo, ["cvap_nh_white", "cvap_white", "cvap_white_nh"]),
    "other": pick_existing(geo, ["cvap_other"]),
}

print("Detected:")
print(" total_total_col:", total_total_col)
print(" vap_total_col  :", vap_total_col)
print(" cvap_total_col :", cvap_total_col)
print(" total_by_group :", total_by_group)
print(" vap_by_group   :", vap_by_group)
print(" cvap_by_group  :", cvap_by_group)


Detected:
 total_total_col: total_pop
 vap_total_col  : vap_total
 cvap_total_col : cvap_total
 total_by_group : {'hisp': 'total_hisp', 'black': None, 'white': None, 'other': 'total_other'}
 vap_by_group   : {'hisp': 'vap_hisp', 'black': 'vap_nh_black', 'white': 'vap_nh_white', 'other': 'vap_other'}
 cvap_by_group  : {'hisp': 'cvap_hisp', 'black': 'cvap_nh_black', 'white': 'cvap_nh_white', 'other': 'cvap_other'}


In [8]:
if cvap_total_col is None:
    raise ValueError("cvap_total not found. Your geo_vtd.parquet must contain cvap_total (or cvap).")

total_count = sum_col(geo, total_total_col) if total_total_col else None
vap_count   = sum_col(geo, vap_total_col) if vap_total_col else None
cvap_count  = sum_col(geo, cvap_total_col)

def compute_group_totals(kind_total_col, kind_map, kind_name):
    """Return (group_totals_dict, denominator_total) or (None, None) if total missing."""
    if kind_total_col is None:
        return None, None

    denom = sum_col(geo, kind_total_col)

    totals = {}
    missing = []
    for _, key in GROUPS:
        c = kind_map.get(key)
        if c is None:
            missing.append(key)
        else:
            totals[key] = sum_col(geo, c)

    # If only 'other' missing, compute residual when possible
    if "other" in missing and all(kind_map.get(k) is not None for k in ["hisp", "black", "white"]):
        totals["other"] = max(0, denom - totals["hisp"] - totals["black"] - totals["white"])
        missing = [m for m in missing if m != "other"]

    if missing:
        print(f"[WARN] Missing {kind_name} by-group columns for: {missing}. Leaving those shares blank.")
    return totals, denom

total_grp, total_den = compute_group_totals(total_total_col, total_by_group, "Total population")
vap_grp, vap_den     = compute_group_totals(vap_total_col, vap_by_group, "VAP")
cvap_grp, cvap_den   = compute_group_totals(cvap_total_col, cvap_by_group, "CVAP")

rows = []
for group_name, key in GROUPS:
    row = {"Racial group": group_name}

    row["Share of total population"] = fmt_pct(total_grp[key] / total_den) if total_grp and total_den and key in total_grp else ""
    row["Share of VAP"]              = fmt_pct(vap_grp[key] / vap_den) if vap_grp and vap_den and key in vap_grp else ""
    row["Share of CVAP"]             = fmt_pct(cvap_grp[key] / cvap_den) if cvap_grp and cvap_den and key in cvap_grp else ""

    rows.append(row)

rows.append({
    "Racial group": "Total count",
    "Share of total population": f"{total_count:,}" if total_count is not None else "",
    "Share of VAP": f"{vap_count:,}" if vap_count is not None else "",
    "Share of CVAP": f"{cvap_count:,}",
})

table_df = pd.DataFrame(rows, columns=["Racial group", "Share of total population", "Share of VAP", "Share of CVAP"])
table_df


[WARN] Missing Total population by-group columns for: ['black', 'white']. Leaving those shares blank.


,Racial group,Share of total population,Share of VAP,Share of CVAP
0,Latino,39.26%,36.16%,31.64%
1,Black,,12.98%,13.17%
2,White,,43.16%,48.28%
3,Other,7.41%,7.70%,6.91%
4,Total count,"29,145,505","21,866,700","19,868,069"


In [9]:
import pandas as pd

def pct(x):
    return f"{x*100:.2f}%"

tot_total = geo["total_pop"].sum()
vap_total = geo["vap_total"].sum()
cvap_total = geo["cvap_total"].sum()

rows = [
    ("Latino", geo["total_hisp"].sum(),     geo["vap_hisp"].sum(),     geo["cvap_hisp"].sum()),
    ("Black",  geo["total_nh_black"].sum(), geo["vap_nh_black"].sum(), geo["cvap_nh_black"].sum()),
    ("White",  geo["total_nh_white"].sum(), geo["vap_nh_white"].sum(), geo["cvap_nh_white"].sum()),
    ("Other",  geo["total_other"].sum(),    geo["vap_other"].sum(),    geo["cvap_other"].sum()),
]

table = pd.DataFrame(rows, columns=["Racial group","total_count","vap_count","cvap_count"])

table["Share of total population"] = table["total_count"] / tot_total
table["Share of VAP"] = table["vap_count"] / vap_total
table["Share of CVAP"] = table["cvap_count"] / cvap_total

# Format for display
table_display = table[["Racial group","Share of total population","Share of VAP","Share of CVAP"]].copy()
for c in ["Share of total population","Share of VAP","Share of CVAP"]:
    table_display[c] = table_display[c].map(pct)

# Totals row like your example
totals_row = pd.DataFrame([{
    "Racial group": "Total count",
    "Share of total population": f"{int(tot_total):,}",
    "Share of VAP": f"{int(vap_total):,}",
    "Share of CVAP": f"{int(cvap_total):,}",
}])

pd.concat([table_display, totals_row], ignore_index=True)


,Racial group,Share of total population,Share of VAP,Share of CVAP
0,Latino,39.26%,36.16%,31.64%
1,Black,13.60%,12.98%,13.17%
2,White,39.75%,43.16%,48.28%
3,Other,7.41%,7.70%,6.91%
4,Total count,"29,145,505","21,866,700","19,868,069"
